In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
SILVER_PATH = r"data/silver"
GOLD_PATH = r"data/gold"

os.makedirs(GOLD_PATH, exist_ok=True)


In [6]:
fin_txn_df = pd.read_csv(r"C:\Users\Rudra\Desktop\smart-transaction-ledger\data\silver\finance_transactions_clean.csv")
user_df = pd.read_csv(r"C:\Users\Rudra\Desktop\smart-transaction-ledger\data\silver\user_db_clean.csv")

In [7]:
fin_txn_df

,transaction_id,user_id,date,amount,merchant_name,payment_mode,category,user_category_new,clean_brand,is_refund
0,TXN0001,U038,2024-03-11,6614.32,Starbucks,UPI,other,Low,Starbucks,0
1,TXN0002,U033,2024-01-31,14567.58,Amazon,Cash,refund,Medium-Low,Amazon,1
2,TXN0003,U013,2024-01-13,12937.22,Local Shop,UPI,shopping,Medium-Low,Local Shop,0
3,TXN0004,U026,2024-02-13,5755.79,Starbucks,Card,shopping,Low,Starbucks,0
4,TXN0005,U036,2024-01-13,29978.48,Apple,UPI,refund,High,Apple,1
...,...,...,...,...,...,...,...,...,...,...
995,TXN0996,U047,2024-01-19,20953.27,DMart,UPI,food,Medium-High,DMart,0
996,TXN0997,U018,2024-03-16,24817.09,OLA,Card,Food & Beverage,High,OLA,0
997,TXN0998,U029,2024-01-06,11386.02,Local Shop,Card,NaN,Medium-Low,Local Shop,0
998,TXN0999,U002,2024-03-18,27224.41,DMart,Card,Food & Beverage,High,DMart,0


In [9]:
user_df

,user_id,city,spending_limit,income_range_clean,risk_score_new
0,U001,BSSR,10000.0,Under 45k,0.1
1,U002,BBSR,15000.0,Over 100k,0.5
2,U003,BBSR,10000.0,Under 45k,0.5
3,U004,BSSR,10000.0,50k - 70k,0.5
4,U005,BSSR,50000.0,Over 100k,0.5
5,U006,BBSR,10000.0,70k - 90k,0.5
6,U007,CTC,50000.0,Over 100k,0.1
7,U008,BBSR,10000.0,50k - 70k,0.5
8,U009,BSSR,15000.0,Under 45k,0.5
9,U010,BSSR,10000.0,Under 45k,0.5


In [8]:
df = fin_txn_df.merge(user_df, on='user_id', how='left')


df['date'] = pd.to_datetime(df['date'])
df['day'] = df['date'].dt.date
df['weekday'] = df['date'].dt.weekday
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)

In [11]:
df

,transaction_id,user_id,date,amount,merchant_name,payment_mode,category,user_category_new,clean_brand,is_refund,city,spending_limit,income_range_clean,risk_score_new,day,weekday,is_weekend
0,TXN0001,U038,2024-03-11,6614.32,Starbucks,UPI,other,Low,Starbucks,0,BBSR,50000.0,70k - 90k,0.5,2024-03-11,0,0
1,TXN0002,U033,2024-01-31,14567.58,Amazon,Cash,refund,Medium-Low,Amazon,1,CTC,50000.0,70k - 90k,0.1,2024-01-31,2,0
2,TXN0003,U013,2024-01-13,12937.22,Local Shop,UPI,shopping,Medium-Low,Local Shop,0,BSSR,50000.0,45k - 55k,0.1,2024-01-13,5,1
3,TXN0004,U026,2024-02-13,5755.79,Starbucks,Card,shopping,Low,Starbucks,0,CTC,15000.0,45k - 55k,0.1,2024-02-13,1,0
4,TXN0005,U036,2024-01-13,29978.48,Apple,UPI,refund,High,Apple,1,BSSR,25000.0,50k - 70k,0.1,2024-01-13,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,TXN0996,U047,2024-01-19,20953.27,DMart,UPI,food,Medium-High,DMart,0,CTC,10000.0,Over 100k,0.3,2024-01-19,4,0
996,TXN0997,U018,2024-03-16,24817.09,OLA,Card,Food & Beverage,High,OLA,0,BSSR,20000.0,Over 100k,0.1,2024-03-16,5,1
997,TXN0998,U029,2024-01-06,11386.02,Local Shop,Card,NaN,Medium-Low,Local Shop,0,BBSR,15000.0,Under 45k,0.5,2024-01-06,5,1
998,TXN0999,U002,2024-03-18,27224.41,DMart,Card,Food & Beverage,High,DMart,0,BBSR,15000.0,Over 100k,0.5,2024-03-18,0,0


In [13]:
user_agg = df.groupby('user_id').agg(

    # Spending Behavior
    total_spent=('amount', 'sum'),
    avg_transaction=('amount', 'mean'),
    max_transaction=('amount', 'max'),
    transaction_count=('transaction_id', 'count'),

    # Time Behavior
    active_days=('day', 'nunique'),
    weekend_spending=('amount', lambda x: x[df.loc[x.index, 'is_weekend'] == 1].sum()),

    # Payment Behavior
    upi_count=('payment_mode', lambda x: (x == 'UPI').sum()),
    cash_count=('payment_mode', lambda x: (x == 'Cash').sum()),

    # Category Behavior
    most_used_category=('category', lambda x: x.mode()[0] if not x.mode().empty else 'Unknown'),
    category_diversity=('category', 'nunique')

).reset_index()

In [14]:
user_agg

,user_id,total_spent,avg_transaction,max_transaction,transaction_count,active_days,weekend_spending,upi_count,cash_count,most_used_category,category_diversity
0,U001,255869.15,17057.943333,26500.75,15,15,350.40,7,3,Food & Beverage,7
1,U002,268640.66,13432.033000,27224.41,20,20,96501.03,9,5,Food & Beverage,8
2,U003,252994.33,13315.491053,29174.50,19,17,19659.72,5,10,refund,7
3,U004,385418.03,17519.001364,29232.21,22,18,79375.97,10,7,grocery,8
4,U005,283306.45,16665.085294,26715.76,17,14,54396.74,6,5,grocery,8
5,U006,384825.99,14800.999615,29678.83,26,23,38307.97,10,7,shopping,8
6,U007,399118.81,15350.723462,28512.44,26,23,112028.18,7,10,grocery,8
7,U008,328357.28,16417.864000,28382.11,20,15,97247.61,8,3,Food & Beverage,8
8,U009,403331.19,17536.138696,29712.06,23,19,77599.60,6,7,Food & Beverage,8
9,U010,249449.61,14673.506471,29098.01,17,17,108172.46,5,9,food,8


In [15]:
# Spending per day
user_agg['spending_per_day'] = user_agg['total_spent'] / user_agg['active_days']

# Payment ratios
user_agg['upi_ratio'] = user_agg['upi_count'] / user_agg['transaction_count']
user_agg['cash_ratio'] = user_agg['cash_count'] / user_agg['transaction_count']

# Merge back user info
user_agg = user_agg.merge(user_df, on='user_id', how='left')


In [21]:
user_agg.columns.tolist()

['user_id',
 'total_spent',
 'avg_transaction',
 'max_transaction',
 'transaction_count',
 'active_days',
 'weekend_spending',
 'upi_count',
 'cash_count',
 'most_used_category',
 'category_diversity',
 'spending_per_day',
 'upi_ratio',
 'cash_ratio',
 'city',
 'spending_limit',
 'income_range_clean',
 'risk_score_new']

In [16]:
user_agg

,user_id,total_spent,avg_transaction,max_transaction,transaction_count,active_days,weekend_spending,upi_count,cash_count,most_used_category,category_diversity,spending_per_day,upi_ratio,cash_ratio,city,spending_limit,income_range_clean,risk_score_new
0,U001,255869.15,17057.943333,26500.75,15,15,350.40,7,3,Food & Beverage,7,17057.943333,0.466667,0.200000,BSSR,10000.0,Under 45k,0.1
1,U002,268640.66,13432.033000,27224.41,20,20,96501.03,9,5,Food & Beverage,8,13432.033000,0.450000,0.250000,BBSR,15000.0,Over 100k,0.5
2,U003,252994.33,13315.491053,29174.50,19,17,19659.72,5,10,refund,7,14882.019412,0.263158,0.526316,BBSR,10000.0,Under 45k,0.5
3,U004,385418.03,17519.001364,29232.21,22,18,79375.97,10,7,grocery,8,21412.112778,0.454545,0.318182,BSSR,10000.0,50k - 70k,0.5
4,U005,283306.45,16665.085294,26715.76,17,14,54396.74,6,5,grocery,8,20236.175000,0.352941,0.294118,BSSR,50000.0,Over 100k,0.5
5,U006,384825.99,14800.999615,29678.83,26,23,38307.97,10,7,shopping,8,16731.564783,0.384615,0.269231,BBSR,10000.0,70k - 90k,0.5
6,U007,399118.81,15350.723462,28512.44,26,23,112028.18,7,10,grocery,8,17352.991739,0.269231,0.384615,CTC,50000.0,Over 100k,0.1
7,U008,328357.28,16417.864000,28382.11,20,15,97247.61,8,3,Food & Beverage,8,21890.485333,0.400000,0.150000,BBSR,10000.0,50k - 70k,0.5
8,U009,403331.19,17536.138696,29712.06,23,19,77599.60,6,7,Food & Beverage,8,21227.957368,0.260870,0.304348,BSSR,15000.0,Under 45k,0.5
9,U010,249449.61,14673.506471,29098.01,17,17,108172.46,5,9,food,8,14673.506471,0.294118,0.529412,BSSR,10000.0,Under 45k,0.5
